# SFT Training with Qwen 2.5 7b

The Notebook requires GPU for execution. This can be run on Runpod or any other cloud GPU provider.



## Installation

In [ ]:
!python -m pip install --upgrade pip -q
!pip install uv -qU
!uv pip install unsloth -qU --system

In [ ]:
# AFTER INSTALLATION - RESTART THE KERNEL

In [ ]:
#!uv pip freeze > requirements-vllm-unsloth-ddp.txt --system

Hugging face setup

In [ ]:
from huggingface_hub import HfFolder, login

# Check if a token is already saved
if HfFolder.get_token() is None:
    login()  # Will prompt only if not logged in

Force Hugging Face to store downloaded models/tokenizers in '/workspace' instead of the default cache location.

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace"
os.environ["HF_HUB_CACHE"] = "/workspace/hub" # (recommended) override just the repo cache
print(os.environ["HF_HOME"])

## Fine-tuning

In [ ]:
# # Base/Instruct Models

enable_thinking = False # set true if using thinking with Qwen 3 models.

test_run = False # to only run a limited set of dataset rows.
# -------
max_seq_length = 1024
dtype = None # unsloth will set this automatically
load_in_4bit = False # Use 4bit quantization to reduce memory usage. Can be False.
load_in_8bit = False
# -------

# # Dataset
## To use a synthetic dataset
#ft_dataset_name = "nijumich/recipieNLG_V1"            

# Question / evaluation criteria / answer column names (adjust to your dataset)
q_column = "input"
c_column = "output"
# c_column = "evaluation_criteria" # swap to "answer" for a hacky way to use a ground truth as criteria (def flawed)
a_column = "output"
# a_column = "evaluation_criteria" # can just use evaluation_criteria if dataset has none.

different_eval_dataset = True


In [ ]:
# Helper to clear cuda without restarting the kernel.
from unsloth import FastLanguageModel
import torch
import gc, inspect, sys

import warnings
warnings.filterwarnings( "ignore", message="Trainer.tokenizer is now deprecated. You should use Trainer.tokenizer instead.", category=UserWarning, )

def clear_old_model_refs():
    """
    Delete `teacher` `model` and `tokenizer` (if they exist) from the caller’s
    local *and* global scope, then garbage-collect and free GPU cache.
    """

    # ── figure out the caller’s frame ────────────────────────────
    frm = inspect.currentframe().f_back
    caller_locals  = frm.f_locals
    caller_globals = frm.f_globals

    for var in ("teacher", "model", "tokenizer"):
        if var in caller_locals:
            try:
                del caller_locals[var]
                if var in sys.modules:   # rarely needed
                    del sys.modules[var]
                print(f"deleted local  {var}")
            except Exception as e:
                print(f"could not delete local {var}: {e}")

        if var in caller_globals:
            try:
                del caller_globals[var]
                print(f"deleted global {var}")
            except Exception as e:
                print(f"could not delete global {var}: {e}")

    # ── Python & CUDA cleanup ───────────────────────────────────
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU cache cleared.")

In [ ]:
clear_old_model_refs()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_slug,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    load_in_8bit = load_in_8bit,
    use_gradient_checkpointing="unsloth",
    # fast_inference=False # ADD THIS IN TO USE VLLM FOR AUTOREGRESSIVE FORWARD PASSES
    # cache_dir = "./" # not necessary if you have already downloaded the model with vllm as HF_HOME is set
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf, if you're not already logged in to hf
)

print(tokenizer.padding_side)

In [ ]:
print(model)

In [ ]:
rank=8 # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
lora_alpha=16 # Should be the square root of the smallest matrix dimension above.

model = FastLanguageModel.get_peft_model(
    model,
    r = rank,
    
    # OR via specific module names
    target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj", # don't train these if it's a MoE (not tested with unsloth, but should work for qwen3)
        ],
    # OR all linear layers (also not recommended)
    # target_modules = ["all-linear"], # to train all linear layers

    # modules_to_save = ["lm_head","embed_tokens"], # to full train embeddings
    lora_alpha = lora_alpha,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    # full_finetuning = False,
    random_state = 3407,
    use_rslora = True,  # rank stabilized LoRA
)

In [ ]:
# print(model)

In [ ]:
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset
offline = True
if offline: 
    dataset = load_dataset("json", data_files={"train": "/workspace/ADVANCED-fine-tuning/fine-tune/datasets/train_gold_80k.jsonl",
    "validation": "/workspace/ADVANCED-fine-tuning/fine-tune/datasets/val_3k.jsonl"})
    
    train_ds = dataset["train"]
    eval_ds = dataset["validation"]
    #test_ds = dataset["test"]
    ft_train_data = train_ds#train_ds.select(range(100))  # Take only first 100 rows
    ft_eval_data = eval_ds#eval_ds.select(range(100))  # Take only first 100 rows
    #ft_test_data = test_ds.select(range(100))  # Take only first 100 rows

else :
    ft_data = load_dataset(ft_dataset_name)
    ft_train_data = ft_data["train"]

    if different_eval_dataset is not None:
        ft_eval_data = different_eval_dataset
    else:
        if "validation" in ft_data:
            ft_eval_data = ft_data["validation"]
        else:
            ft_eval_data = None
            print("No eval data")

In [ ]:
print(ft_train_data)

In [ ]:
print(ft_train_data['input'][0])

In [ ]:
# To down-select data.
if test_run: ft_train_data = ft_train_data.select(range(10000))
if test_run: ft_eval_data = ft_eval_data.select(range(1000))

In [ ]:
def formatting_func(batch):
    """Convert a *batch* of rows to a list[str] of chat-formatted prompts."""
    out = []
    
    # batch[q_column] is a list; iterate over it
    for title_and_ingredients,directions in zip(batch[q_column], batch[a_column]):

        messages = [
            {"role": "system", 
             "content": "You are a culinary assistant. "
                    "Write step-by-step cooking directions using the given title and ingredients. "
                    "Use all relevant ingredients"
                    "Do NOT repeat the ingredient list. "
                    "Use complete sentences."
                    "Use numbered steps with action verbs."},
            {"role": "user", "content": title_and_ingredients},
            {"role": "assistant", "content": directions}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=enable_thinking,
        )

        # Ensures two bos tokens aren't added, because the later tokenisation adds one!
        bos = tokenizer.bos_token or "<bos>"
        if text.startswith(bos):
            text = text[len(bos):]

        out.append(text)

    return out        # ← length == batch size

In [ ]:
print(formatting_func(ft_eval_data)[2])

In [ ]:
from trl import SFTTrainer, SFTConfig
from datetime import datetime

per_device_train_batch_size = 64 #4 # reduce if you run out of VRAM.
gradient_accumulation_steps = 1 #int(32 / per_device_train_batch_size)
epochs = 1
learning_rate = 1e-4  # for a 1B model go for 2e-4, for 8B go for 2e-5, for 3B go for 1e-4, 30B go for 5e-6

# Get current timestamp
current_timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')

if 'ft_dataset_name' in globals() or 'ft_dataset_name' in locals():
    # The variable is defined, proceed with your logic
    if ft_dataset_name is not None:
        run_name = f"{model_slug.split('/')[-1]}-{ft_dataset_name.split('/')[-1][:15]}-{epochs}ep-{current_timestamp}"
    else:
        print("ft_dataset_name is not defined and a run name cannot be set to include it. Check and rerun this cell")
else:
    run_name = f"{model_slug.split('/')[-1]}-{dataset_name.split('/')[-1][:15]}-{epochs}ep-{current_timestamp}"

In [ ]:
# ──────────────────
# Decide if we evaluate
# ──────────────────
do_eval = ft_eval_data is not None          # True ⇢ we have a dataset
if do_eval:
    print(f"Will run evaluation using validation dataset")
else:
    print(f"Will not run evaluation, as no dataset was passed")

# ──────────────────
# Build training_args
# ──────────────────
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    per_device_train_batch_size = per_device_train_batch_size,
    per_device_eval_batch_size  = 2 ,#per_device_train_batch_size,
    gradient_accumulation_steps = gradient_accumulation_steps,
    num_train_epochs            = epochs,
    #max_steps=30,   # uncomment for shorter run
    
    logging_strategy = "steps",
    logging_dir      = f"logs/{model_slug.split('/')[-1]}",
    eval_strategy    = "steps",
    logging_steps    = 10, #1       # cannot be fractional (0.05 invalid)
    eval_steps       = 250, #0.02,      # ⚠️ must be int steps, not a fraction
    
    bf16  = is_bfloat16_supported(),
    fp16  = not is_bfloat16_supported(),
    report_to = "tensorboard",
    seed      = 3407,
    output_dir = "outputs",

    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": True},
    remove_unused_columns = True,
    lr_scheduler_type     = "cosine",
)

#### Fine-tuning

In [ ]:
num_gpus = torch.cuda.device_count()
gpu_tag = f"{num_gpus}gpu" if num_gpus > 0 else "cpu"

run_name = f"{run_name}-ft-{gpu_tag}"
print(f"Setting up for run: {run_name}")

training_args.run_name = run_name
training_args.logging_dir = f"./logs/{run_name}"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ft_train_data,
    eval_dataset=ft_eval_data,
    args=training_args,
    formatting_func=formatting_func
)

In [ ]:
print(trainer.train_dataset)

In [ ]:
from unsloth.chat_templates import train_on_responses_only 

TEMPLATES = {
    "llama": (
        "<|start_header_id|>user<|end_header_id|>\n\n",
        "<|start_header_id|>assistant<|end_header_id|>\n\n",
    ),
    "gemma": (
        "<start_of_turn>user\n",
        "<start_of_turn>model\n",
    ),
    "qwen": (
        "<|im_start|>user\n",
        "<|im_start|>assistant\n", # No thinking
    ),
    "mistral": (
        "[INST]",
        "[/INST]",
    )
}

instruction_tag, response_tag = TEMPLATES["qwen"]   # ← change if needed

# masks everything between the instruction_part and response_part
trainer = train_on_responses_only(
    trainer,
    instruction_part = instruction_tag,
    response_part = response_tag,
    # force_match=False # comment out to set true for a cleaner masking
)

In [ ]:
print(trainer.train_dataset)

In [ ]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

Check if Masking is working

In [ ]:
# Test one example from your processed dataset
sample = trainer.train_dataset[0]
decoded_labels = tokenizer.decode([l for l in sample['labels'] if l != -100])
decoded_input = tokenizer.decode(sample['input_ids'])

print("--- FULL INPUT ---")
print(decoded_input)
print("\n--- WHAT THE MODEL ACTUALLY LEARNS (LABELS) ---")
print(decoded_labels)

# VERIFY: If decoded_labels contains ingredients, your assistant_start_idx is wrong.

#### Start training

In [ ]:
# Check memory
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Final memory results
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### Save and/or Push the Model to Hub

In [ ]:
print(run_name)

In [ ]:
# # # Manually shorten the run name and just re-run the push, if needed.
# run_name = "ddp-demo-{num-gpus}"

In [ ]:
# # Merge to 16bit (RECOMMENDED, should merge to a dequantized base model for best accuracy)
org = "nijumich"
# print(f"Saving and pushing as {run_name} and {org}/{run_name}")

# Just SAVE locally
#if True: model.save_pretrained_merged(f"{run_name}", tokenizer, save_method = "merged_16bit",)

# # # Save locally AND push to hub
if True: model.push_to_hub_merged(f"{org}/{run_name}", 
                                  tokenizer, 
                                  save_method = "merged_16bit",
                                   token = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

# Not a best option to expose the hf token here 

In [ ]:
# Print the run name
print(run_name)

Notes: 

    This notebook is based on a Notebook by [Trelis Research](https://trelis.com/about).
    Available at [Trelis.com/ADVANCED-fine-tuning]().